# Notebook B: chia dữ liệu, mô hình và câu hỏi thứ ba

**Đề tài:** Dự đoán kết quả quần vợt thật sự đạt bao nhiêu khi chỉ dùng thông tin trước trận và số liệu trong trận thổi phồng kết quả đến mức nào

**Sinh viên B** chạy sau khi Notebook A đã tạo file parquet, `sql/queries.sql` và `manifest.json`. Notebook này không cần kernel của Notebook A.

## Kiểm tra bàn giao

- **Làm gì:** đọc manifest và so mã băm của file parquet.
- **Vì sao:** bảo đảm hai notebook dùng đúng cùng một dữ liệu.
- **Kiểm tra:** ô in `handover ok`.

In [ ]:
# run once per machine, then comment out
# %pip install pandas numpy duckdb pyarrow scikit-learn lightgbm xgboost matplotlib scipy
import os, json, hashlib, numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore')
m = json.load(open('data/processed/manifest.json'))
assert hashlib.md5(open(m['file'], 'rb').read()).hexdigest() == m['md5'], 'parquet differs from the manifest, rerun Notebook A'
os.makedirs('report', exist_ok=True); print('handover ok', m['rows'], 'rows')

## Bước 4. Bảng đặc trưng và chia dữ liệu

- **Làm gì:** chạy lại `sql/queries.sql`, lấy bảng `feat`, chia tập huấn luyện và kiểm thử theo giao thức của đề tài.
- **Vì sao:** cách chia quyết định con số có đáng tin hay không; dòng `assert` là phép kiểm tra rò rỉ.
- **Kiểm tra:** ghi số dòng hai tập và số đặc trưng vào bảng mô tả thực nghiệm.

In [ ]:
# Step 4: forward split by year: train on 2008 to 2022, validate on 2023, test on 2024 to 2026; only features known before the match are used
import duckdb
con = duckdb.connect(); con.execute(open('sql/queries.sql').read())
feat = con.execute("SELECT * FROM feat ORDER BY date, match_id").df(); feat = pd.get_dummies(feat, columns=['surface'], dtype=int)
PRE = ['rank_diff', 'log_points_diff', 'age_diff', 'height_diff', 'elo_diff', 'surface_elo_diff', 'played_diff', 'form10_diff', 'h2h_wins_diff', 'best_of_5'] + [c for c in feat.columns if c.startswith('surface_') and c != 'surface_elo_diff']
POST = [c for c in feat.columns if c.startswith('inmatch_')]                       # used only in RQ3 to show what leakage does
feat[PRE + POST] = feat[PRE + POST].fillna(0); X_cols = PRE; TARGET = 'y'
train, valid, test = feat[feat.year <= 2022], feat[feat.year == 2023], feat[feat.year >= 2024]
X_tr, y_tr, X_te, y_te = train[X_cols], train[TARGET], test[X_cols], test[TARGET]
assert train.date.max() < valid.date.min() and valid.date.max() <= test.date.min()       # leakage check
print(len(train), len(valid), len(test), len(PRE), 'pre-match features', len(POST), 'in-match statistics kept aside')

## Bước 4b. Hiểu tập huấn luyện và tập kiểm thử

Ba ô dưới đây kiểm tra hai tập có giống nhau không và đặc trưng nào mạnh nhất, trước khi huấn luyện bất kỳ mô hình nào.

### 4b.1 Mục tiêu ở hai tập

- **Làm gì:** so phân phối của mục tiêu giữa tập huấn luyện và tập kiểm thử.
- **Vì sao:** hai phân phối lệch nhau giải thích vì sao mô hình cây ngoại suy kém.

> **Nhận xét của nhóm (bắt buộc, một câu có số):** ...

In [ ]:
import matplotlib.pyplot as plt
def bar_chart(labels, values, ylabel, path, highlight=None, fmt='{:.1f}'):
    fig, ax = plt.subplots(figsize=(6.5, 3.2))
    colors = ['#F4A261' if i == highlight else '#1B6B6D' for i in range(len(values))]
    bars = ax.bar([str(l) for l in labels], values, color=colors)
    ax.bar_label(bars, labels=[fmt.format(v) for v in values], fontweight='bold', padding=2, fontsize=8 if len(values) > 7 else 10)
    ax.set_ylabel(ylabel); ax.set_ylim(0, max(values) * 1.2)
    ax.spines[['top', 'right']].set_visible(False)          # no top and right spines
    if len(values) > 7: ax.tick_params(axis='x', labelsize=7, rotation=30); plt.setp(ax.get_xticklabels(), ha='right')
    fig.tight_layout(); fig.savefig(path, dpi=300); plt.show()

import matplotlib.pyplot as plt
def nice(v, label):                                     # readable numbers on the bars
    v = np.asarray(v, dtype=float); m = np.nanmax(np.abs(v))
    return (v / 1e6, label + ' (million)', '{:.2f}') if m >= 1e6 else (v / 1e3, label + ' (thousand)', '{:.1f}') if m >= 1e4 else (v, label, '{:.2f}' if m < 100 else '{:.0f}')
ytr, yte = pd.Series(np.asarray(y_tr, dtype=float)), pd.Series(np.asarray(y_te, dtype=float)); BINARY = ytr.nunique() <= 2
print('train rows', len(ytr), 'test rows', len(yte), 'features', X_tr.shape[1])
if BINARY: print('positive share: train %.3f, test %.3f' % (ytr.mean(), yte.mean()))
else:
    fig, ax = plt.subplots(figsize=(6.5, 3)); lo, hi = min(ytr.quantile(.01), yte.quantile(.01)), max(ytr.quantile(.99), yte.quantile(.99)); bins = np.linspace(lo, hi, 50)
    ax.hist(ytr.clip(lo, hi), bins=bins, density=True, alpha=.6, color='#1B6B6D', label='Train'); ax.hist(yte.clip(lo, hi), bins=bins, density=True, alpha=.6, color='#F4A261', label='Test')
    ax.legend(frameon=False); ax.set_xlabel('Target'); ax.set_ylabel('Density'); ax.spines[['top', 'right']].set_visible(False); fig.tight_layout(); fig.savefig('report/fig_split_target.png', dpi=200); plt.show()
    print(pd.DataFrame({'train': ytr.describe(), 'test': yte.describe()}).round(3))

### 4b.2 Đặc trưng lệch giữa hai tập

- **Làm gì:** độ lệch trung bình chuẩn hoá của từng đặc trưng giữa hai tập.
- **Kiểm tra:** đặc trưng lệch trên 0,5 độ lệch chuẩn là nguồn rủi ro, nêu trong mục Limitations.

> **Nhận xét của nhóm (bắt buộc, một câu có số):** ...

In [ ]:
A_, B_ = X_tr.astype(float), X_te.astype(float); sd = A_.std().replace(0, np.nan)
shift = ((B_.mean() - A_.mean()) / sd).abs().sort_values(ascending=False).rename('abs_standardised_mean_difference')
shift.to_csv('report/table_split_shift.csv'); display(shift.head(10).round(3).to_frame())
print('features whose test mean lies more than 0.5 training standard deviations away:', int((shift > 0.5).sum()), 'of', len(shift))

### 4b.3 Sức mạnh của từng đặc trưng

- **Làm gì:** tương quan hạng giữa từng đặc trưng và mục tiêu trên tập huấn luyện.
- **Vì sao:** nếu một đặc trưng đã gần đủ, mô hình phức tạp sẽ không hơn nhiều, và đó là thông tin để chọn câu hỏi.

> **Nhận xét của nhóm (bắt buộc, một câu có số):** ...

In [ ]:
sig = A_.sample(min(len(A_), 20_000), random_state=0); ys = pd.Series(np.asarray(y_tr, dtype=float), index=X_tr.index).loc[sig.index]
strength = sig.corrwith(ys, method='spearman').abs().sort_values(ascending=False).rename('abs_rank_correlation_with_target')
strength.to_csv('report/table_feature_strength.csv'); display(strength.head(10).round(3).to_frame())
print('strongest single feature:', strength.index[0], round(float(strength.iloc[0]), 3), '-> a model must beat what this feature already gives')

## Bước 6a. So sánh mô hình (RQ2)

- **Làm gì:** huấn luyện mười mô hình với năm seed trên cùng cách chia.
- **Vì sao:** so nhiều họ mô hình để kết luận không phụ thuộc một kiểu mô hình; năm seed cho độ lệch chuẩn.
- **Kiểm tra:** chênh lệch nhỏ hơn độ lệch chuẩn thì chưa được kết luận mô hình nào hơn. Bảng lưu ở `report/table_rq2.csv`.

In [ ]:
# Step 6a: ten classifiers from four families, five seeds, trained on 2008 to 2022 and tested on 2024 to 2026, pre-match features only; probability metrics because one class is rare
import numpy as np, pandas as pd, time
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
sc = lambda m: make_pipeline(StandardScaler(), m)       # scaling for linear, distance and neural models
models = {
    'Logistic':       lambda s: sc(LogisticRegression(max_iter=500, random_state=s)),
    'Linear SVM':     lambda s: sc(LinearSVC(C=1.0, random_state=s)),
    'k-NN':           lambda s: sc(KNeighborsClassifier(n_neighbors=15)),
    'Decision Tree':  lambda s: DecisionTreeClassifier(max_depth=12, min_samples_leaf=5, random_state=s),
    'Random Forest':  lambda s: RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=s),
    'Extra Trees':    lambda s: ExtraTreesClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=s),
    'HistGB':         lambda s: HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, random_state=s),
    'LightGBM':       lambda s: LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, subsample=0.8,
                                               subsample_freq=1, colsample_bytree=0.8, random_state=s, verbose=-1),
    'XGBoost':        lambda s: XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6, subsample=0.8,
                                              colsample_bytree=0.8, tree_method='hist', random_state=s),
    'Small MLP':      lambda s: sc(MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=200, early_stopping=True,
                                                 random_state=s)),
}
def proba(m, X):                                        # probability of the positive class for every model family
    if hasattr(m, 'predict_proba'): return m.predict_proba(X)[:, 1]
    return 1 / (1 + np.exp(-m.decision_function(X)))
rows, fitted = [], {}
for name, make in models.items():
    aucs, aps, briers, secs = [], [], [], []
    for seed in range(5):
        m = make(seed); t0 = time.perf_counter(); m.fit(X_tr, y_tr); secs.append(time.perf_counter() - t0)
        p = proba(m, X_te); aucs.append(roc_auc_score(y_te, p)); aps.append(average_precision_score(y_te, p))
        briers.append(brier_score_loss(y_te, p))
    fitted[name] = m                                    # keep the last fitted model of each family for Step 6b
    rows.append([name, np.mean(aucs), np.std(aucs), np.mean(aps), np.mean(briers), np.mean(secs)])
res = pd.DataFrame(rows, columns=['model', 'ROC_AUC', 'sd', 'PR_AUC', 'Brier', 'train_s'])
res.to_csv('report/table_rq2.csv', index=False); print(res.round(4))

### Hình RQ2

- **Làm gì:** vẽ kết quả các mô hình, tô màu nhấn mô hình tốt nhất.
- **Kiểm tra:** số trên cột khớp với `report/table_rq2.csv`.

In [ ]:
import matplotlib.pyplot as plt
def bar_chart(labels, values, ylabel, path, highlight=None, fmt='{:.1f}'):
    fig, ax = plt.subplots(figsize=(6.5, 3.2))
    colors = ['#F4A261' if i == highlight else '#1B6B6D' for i in range(len(values))]
    bars = ax.bar([str(l) for l in labels], values, color=colors)
    ax.bar_label(bars, labels=[fmt.format(v) for v in values], fontweight='bold', padding=2, fontsize=8 if len(values) > 7 else 10)
    ax.set_ylabel(ylabel); ax.set_ylim(0, max(values) * 1.2)
    ax.spines[['top', 'right']].set_visible(False)          # no top and right spines
    if len(values) > 7: ax.tick_params(axis='x', labelsize=7, rotation=30); plt.setp(ax.get_xticklabels(), ha='right')
    fig.tight_layout(); fig.savefig(path, dpi=300); plt.show()

bar_chart(res.model, res.iloc[:, 1], res.columns[1], 'report/fig_rq2_models.png', highlight=int(res.iloc[:, 1].idxmax()), fmt='{:.2f}')

## Cổng khả thi: có nên xoay câu hỏi không

- **Làm gì:** so mô hình tốt nhất với mốc đơn giản và in ra một trong ba kết luận.
- **Vì sao:** nếu mô hình không hơn mốc hoặc bài toán quá dễ, nhóm xoay câu hỏi ngay ở đây thay vì cố chỉnh mô hình. Xoay câu hỏi nghĩa là đổi điều được hỏi cho khớp với điều dữ liệu trả lời được, không phải chọn lại số liệu cho đẹp.

**Các hướng xoay của đề tài này:**

1. Mô hình chỉ hơn quy tắc khoảng hai điểm: lấy phép thử rò rỉ làm kết quả chính.
2. Nếu muốn mức cải thiện lớn hơn: chỉ xét Grand Slam, nơi cả mô hình và quy tắc đều đúng hơn 70%.
3. Nếu phản biện đòi so với tỷ lệ cược: nêu là hướng làm tiếp vì dữ liệu này không có tỷ lệ cược.

**Quyết định của nhóm (giữ câu hỏi hay xoay theo hướng nào, vì sao):** ...

In [ ]:
best = res.sort_values('ROC_AUC').iloc[-1]; spread = res.ROC_AUC.max() - res.ROC_AUC.min()
print('best model %s: ROC-AUC %.4f, PR-AUC %.4f | spread between models %.4f' % (best.model, best.ROC_AUC, best.PR_AUC, spread))
verdict = 'TOO EASY: every model is near the ceiling, so the comparison says little; explain why the task is easy or use a harder protocol (RQ3)' if best.ROC_AUC >= 0.995 else 'GOOD: there is a clear signal and room to compare models' if best.ROC_AUC >= 0.75 else 'WEAK: the signal is weak, pivot the target or the features (see the options above) and report this result honestly'
print(verdict); pd.Series({'best_model': best.model, 'ROC_AUC': best.ROC_AUC, 'PR_AUC': best.PR_AUC, 'spread': spread, 'verdict': verdict}).to_csv('report/table_feasibility_gate.csv')

## Bước 6b. Câu hỏi thứ ba (RQ3)

- **Làm gì:** Huấn luyện lại bốn mô hình đại diện với thêm sáu số liệu trong trận để đo mức thổi phồng. vì aciids là hội nghị hạng b, notebook_b có thêm phần nâng cao r0 đến r6: mọi lựa chọn được làm trên 20% cuối của tập huấn luyện, lightgbm được tinh chỉnh với 12 cấu hình, một mô hình trộn bốn họ được so với mô hình đơn lẻ, chênh lệch giữa hai phương pháp chỉ được kết luận khi khoảng bootstrap 95% theo khối không chứa số 0, dự đoán đi kèm hiệu chỉnh xác suất, và permutation importance được lặp với ba seed để kiểm tra độ ổn định.
- **Vì sao:** đây là đóng góp chính của bài; mỗi bảng `report/table_rq3_*.csv` ứng với một ý trong mục Results.
- **Kiểm tra:** đọc từng bảng in ra và viết một câu kết luận có số cho mỗi bảng. Nếu kết quả trái dự kiến trong đề cương thì báo đúng như vậy và tìm nguyên nhân, không chỉnh code cho ra kết quả mong muốn.

In [ ]:
# Step 6b: RQ3, how much accuracy comes from statistics of the match itself, and do ten models beat a rating rule when only pre-match information is allowed?
from sklearn.linear_model import LogisticRegression
acc = lambda y, p: float(((p >= 0.5) == y).mean())
rule_rank = (test.rank_diff < 0).astype(float).to_numpy(); elo_rule = LogisticRegression().fit(train[['elo_diff', 'surface_elo_diff']], y_tr); pe = elo_rule.predict_proba(test[['elo_diff', 'surface_elo_diff']])[:, 1]
rows = [['Rule: better-ranked player', 1, acc(y_te, rule_rank), np.nan, np.nan], ['Rule: two ratings', 2, acc(y_te, pe), roc_auc_score(y_te, pe), brier_score_loss(y_te, pe)]]
for name, m in fitted.items(): p = np.clip(proba(m, X_te), 0, 1); rows.append([name + ', pre-match', len(PRE), acc(y_te, p), roc_auc_score(y_te, p), brier_score_loss(y_te, p)])
for name in ['Logistic', 'Random Forest', 'LightGBM', 'Small MLP']:               # the same models when statistics of the match itself are added
    p = np.clip(proba(models[name](0).fit(train[PRE + POST], y_tr), test[PRE + POST]), 0, 1); rows.append([name + ', with in-match statistics', len(PRE + POST), acc(y_te, p), roc_auc_score(y_te, p), brier_score_loss(y_te, p)])
leak = pd.DataFrame(rows, columns=['method', 'n_features', 'accuracy', 'ROC_AUC', 'Brier']); leak.to_csv('report/table_rq3_leakage.csv', index=False); print(leak.round(3))
# (2) by surface and by tournament level: best model against the rating rule
best = res.sort_values('ROC_AUC').model.iloc[-1]; d = test.assign(p=np.clip(proba(fitted[best], X_te), 0, 1), q=pe); surf = [c for c in feat.columns if c.startswith('surface_') and c != 'surface_elo_diff']
d['surface'] = d[surf].idxmax(axis=1).str.replace('surface_', ''); rows = []
for col in ['surface', 'level', 'best_of_5']:
    for v, g in d.groupby(col):
        if len(g) >= 200: rows.append([col, v, len(g), acc(g.y, g.p), acc(g.y, g.q), brier_score_loss(g.y, g.p), brier_score_loss(g.y, g.q)])
seg = pd.DataFrame(rows, columns=['split_by', 'segment', 'matches', 'accuracy_model', 'accuracy_rating_rule', 'Brier_model', 'Brier_rating_rule']); seg.to_csv('report/table_rq3_segments.csv', index=False); print(best); print(seg.round(3))

## Bước 6c. Kiểm tra bổ sung và bảng tham số

- **Làm gì:** chạy phép kiểm tra bổ sung của đề tài và lưu tham số mô hình.
- **Vì sao:** bảng này đi vào phần thảo luận và phụ lục của bài.
- **Kiểm tra:** `report/params_best.json` tồn tại.

In [ ]:
# Step 6c: calibration of the best model, the result by year after training, and parameters
d['bin'] = pd.cut(d.p, np.linspace(0, 1, 11), include_lowest=True); cal = d.groupby('bin', observed=True).agg(matches=('y', 'size'), mean_predicted=('p', 'mean'), observed_win_share=('y', 'mean'))
cal.to_csv('report/table_rq3_calibration.csv'); print(cal.round(3))
by = d.groupby('year').apply(lambda g: pd.Series({'matches': len(g), 'accuracy_model': acc(g.y, g.p), 'accuracy_rating_rule': acc(g.y, g.q), 'accuracy_better_ranked': acc(g.y, (g.rank_diff < 0).astype(float))}))
by.to_csv('report/table_rq3_by_year.csv'); print(by.round(3))
conf = d.assign(sure=pd.cut(np.abs(d.p - 0.5), [0, 0.1, 0.25, 0.5], labels=['toss-up (50 to 60%)', 'leaning (60 to 75%)', 'confident (over 75%)'], include_lowest=True)).groupby('sure', observed=True).apply(
    lambda g: pd.Series({'matches': len(g), 'share_pct': 100 * len(g) / len(d), 'accuracy': acc(g.y, g.p)})); conf.to_csv('report/table_rq3_confidence.csv'); print(conf.round(3))
import json
json.dump({'best': best, 'train_until': 2022, 'test_from': 2024, 'params': {k: str(v) for k, v in models[best](0).get_params().items()}}, open('report/params_best.json', 'w'), indent=2)

### Hình RQ3

- **Làm gì:** vẽ hai hình RQ3 từ các bảng `report/table_rq3_*.csv` bằng hàm `bar_chart` ở trên, theo mẫu Hình 4 và Hình 5 của đề cương.
- **Kiểm tra:** mỗi hình một ý, chữ tiếng Anh, số in đậm trên cột.

In [ ]:
import glob
for f in sorted(glob.glob('report/table_rq3_*.csv')):
    t = pd.read_csv(f); numc = [c for c in t.columns[1:] if pd.api.types.is_numeric_dtype(t[c])]
    print(f, t.shape)
    if 2 <= len(t) <= 14 and numc and t[numc[0]].notna().all() and t[numc[0]].min() >= 0 and t[numc[0]].max() > 0:
        v, lab, fm = nice(t[numc[0]], numc[0]); bar_chart(t.iloc[:, 0].astype(str), v, lab, f.replace('table_', 'fig_').replace('.csv', '.png'), highlight=int(v.argmax()), fmt=fm)

## Phần nâng cao cho hội nghị hạng B (ACIIDS)\n\nPhản biện của hội nghị hạng B thường hỏi năm điều: tham số có được tinh chỉnh công bằng không, có phương pháp nào hơn mô hình đơn lẻ không, chênh lệch có lớn hơn dao động ngẫu nhiên không, dự đoán có kèm độ bất định không, và lời giải thích có ổn định không. Sáu ô dưới đây trả lời lần lượt, mỗi ô lưu một bảng `report/table_rankB_*.csv`.\n\n### R0. Chia trong tập huấn luyện\n\n- **Làm gì:** tách 20% cuối của tập huấn luyện làm phần tinh chỉnh và hiệu chỉnh.\n- **Vì sao:** mọi lựa chọn (tham số, ngưỡng, hiệu chỉnh) phải làm trên phần này, tập kiểm thử chỉ dùng để báo kết quả.\n- **Kiểm tra:** ba con số in ra cộng lại đúng bằng số dòng của hai tập.

In [ ]:
# inner split of the training part in its stored order: the first 80% to fit, the last 20% (later in time, or later groups) to tune and calibrate; the test part is never used for a choice
from lightgbm import LGBMRegressor, LGBMClassifier
IS_REG = False; LG = LGBMRegressor if IS_REG else LGBMClassifier
trs = train.sort_index(); cut = int(0.8 * len(trs)); inner, calib = trs.iloc[:cut], trs.iloc[cut:]
pred = (lambda m, X: np.asarray(m.predict(X), dtype=float)) if IS_REG else (lambda m, X: np.asarray(proba(m, X), dtype=float))
metric = (lambda y, p: float(MAE(y, p))) if IS_REG else (lambda y, p: float(roc_auc_score(y, p)))      # MAE: lower is better; ROC-AUC: higher is better
METRIC = 'MAE' if IS_REG else 'ROC_AUC'; better = (lambda a, b: a < b) if IS_REG else (lambda a, b: a > b)
print('fit on', len(inner), 'rows, tune and calibrate on', len(calib), 'rows, test on', len(test), 'rows; metric:', METRIC)

### R1. Tinh chỉnh tham số\n\n- **Làm gì:** thử 12 cấu hình LightGBM chọn ngẫu nhiên, chấm trên phần hiệu chỉnh, rồi huấn luyện lại cấu hình tốt nhất trên toàn bộ tập huấn luyện.\n- **Vì sao:** so mô hình với tham số mặc định là điểm phản biện hay bắt lỗi.\n- **Kiểm tra:** ghi cấu hình được chọn vào bảng tham số của bài; nếu tinh chỉnh không cải thiện trên tập kiểm thử thì báo đúng như vậy.

In [ ]:
rs = np.random.RandomState(0); rows = []
for i in range(12):                                        # 12 random settings of LightGBM, scored on the calibration part only
    cfg = dict(n_estimators=int(rs.choice([200, 400, 800])), learning_rate=float(rs.choice([0.02, 0.05, 0.1])), num_leaves=int(rs.choice([15, 31, 63])),
               min_child_samples=int(rs.choice([10, 30, 100])), colsample_bytree=float(rs.choice([0.6, 0.8, 1.0])), reg_lambda=float(rs.choice([0.0, 1.0, 10.0])))
    m = LG(subsample=0.8, subsample_freq=1, random_state=0, verbose=-1, **cfg).fit(inner[X_cols], inner[TARGET]); rows.append({**cfg, 'calibration_' + METRIC: metric(calib[TARGET], pred(m, calib[X_cols]))})
tune = pd.DataFrame(rows).sort_values('calibration_' + METRIC, ascending=IS_REG).reset_index(drop=True); tune.to_csv('report/table_rankB_tuning.csv', index=False); display(tune.head(5).round(4))
INTS = ('n_estimators', 'num_leaves', 'min_child_samples'); best_cfg = {k: (int(v) if k in INTS else float(v)) for k, v in tune.iloc[0].drop('calibration_' + METRIC).items()}
tuned = LG(subsample=0.8, subsample_freq=1, random_state=0, verbose=-1, **best_cfg).fit(X_tr, y_tr)
P = {n: pred(m, X_te) for n, m in fitted.items()}; P['LightGBM tuned'] = pred(tuned, X_te)
print('LightGBM on test, default settings: %.4f | tuned on the calibration part: %.4f' % (metric(y_te, P['LightGBM']), metric(y_te, P['LightGBM tuned'])))

### R2. Mô hình trộn bốn họ\n\n- **Làm gì:** lấy trung bình dự đoán của bốn mô hình, mỗi họ một mô hình, danh sách cố định trước khi xem điểm trên tập kiểm thử.\n- **Vì sao:** đây là phương pháp đề xuất tối thiểu của bài; chọn thành phần theo điểm kiểm thử là rò rỉ.\n- **Kiểm tra:** bảng xếp hạng lưu ở `report/table_rankB_methods.csv`.

In [ ]:
LIN = 'Ridge' if IS_REG else 'Logistic'; PARTS = [LIN, 'Random Forest', 'LightGBM tuned', 'Small MLP']      # one model per family, fixed before looking at the test scores
P['Blend of four families'] = np.mean([P[n] for n in PARTS], axis=0)
rk = pd.DataFrame({'method': list(P), METRIC: [metric(y_te, p) for p in P.values()]}).sort_values(METRIC, ascending=IS_REG).reset_index(drop=True)
rk.to_csv('report/table_rankB_methods.csv', index=False); display(rk.round(4)); BEST, SECOND = rk.method.iloc[0], rk.method.iloc[1]

### R3. Khoảng tin cậy bootstrap cho chênh lệch\n\n- **Làm gì:** lấy mẫu lại tập kiểm thử 500 lần theo khối 24 dòng liền nhau, tính chênh lệch metric giữa hai phương pháp.\n- **Vì sao:** chỉ được viết phương pháp A hơn B khi khoảng 95% không chứa số 0.\n- **Kiểm tra:** cột `interval_excludes_zero`; với cặp là False thì trong bài viết hai phương pháp tương đương.

In [ ]:
def boot(a, b, B=500, L=24):                            # paired block bootstrap over the test rows: blocks of 24 neighbouring rows are drawn together
    r = np.random.RandomState(0); y = np.asarray(y_te, dtype=float); n = len(y); d = []
    for _ in range(B):
        i = (r.randint(0, n - L, n // L)[:, None] + np.arange(L)).ravel()
        if not IS_REG and len(np.unique(y[i])) < 2: continue
        d.append(metric(y[i], P[a][i]) - metric(y[i], P[b][i]))
    lo, hi = np.percentile(d, [2.5, 97.5]); return [a, b, float(np.mean(d)), float(lo), float(hi), bool(lo > 0 or hi < 0)]
pairs = [(BEST, SECOND), ('Blend of four families', BEST if BEST != 'Blend of four families' else SECOND), ('LightGBM tuned', 'LightGBM')]
if IS_REG: P['Reference'] = np.asarray(test[NAIVE], dtype=float); pairs.append((BEST, 'Reference'))
bt = pd.DataFrame([boot(a, b) for a, b in pairs], columns=['method_a', 'method_b', 'difference_a_minus_b', 'ci_low', 'ci_high', 'interval_excludes_zero'])
bt.to_csv('report/table_rankB_bootstrap.csv', index=False); display(bt.round(4))

### R4. Hiệu chỉnh xác suất\n\n- **Làm gì:** hiệu chỉnh xác suất bằng hồi quy đẳng điệu trên phần hiệu chỉnh, đo ECE và Brier trước và sau trên tập kiểm thử.\n- **Vì sao:** xác suất dùng để ra quyết định phải khớp với tần suất thật.\n- **Kiểm tra:** ECE sau hiệu chỉnh nhỏ hơn trước; nếu không thì phần hiệu chỉnh và tập kiểm thử khác phân phối, ghi vào thảo luận.

In [ ]:
from sklearn.isotonic import IsotonicRegression
def ece(y, p, bins=10):                                    # expected calibration error
    y = np.asarray(y, dtype=float); b = np.minimum((p * bins).astype(int), bins - 1); return float(sum((b == k).mean() * abs(y[b == k].mean() - p[b == k].mean()) for k in range(bins) if (b == k).any()))
m_in = LG(subsample=0.8, subsample_freq=1, random_state=0, verbose=-1, **best_cfg).fit(inner[X_cols], inner[TARGET])      # fitted without the calibration part
iso = IsotonicRegression(out_of_bounds='clip').fit(pred(m_in, calib[X_cols]), calib[TARGET]); raw_p = pred(m_in, X_te); cal_p = iso.predict(raw_p)
cf = pd.DataFrame([['before calibration', ece(y_te, raw_p), brier_score_loss(y_te, raw_p), roc_auc_score(y_te, raw_p)], ['after isotonic calibration', ece(y_te, cal_p), brier_score_loss(y_te, cal_p), roc_auc_score(y_te, cal_p)]], columns=['probabilities', 'ECE', 'Brier', 'ROC_AUC'])
cf.to_csv('report/table_rankB_uncertainty.csv', index=False); display(cf.round(4))

### R5. Độ ổn định của lời giải thích\n\n- **Làm gì:** tính permutation importance của LightGBM với ba seed, so thứ hạng đặc trưng giữa các seed.\n- **Vì sao:** lời giải thích đổi theo seed thì không được dùng làm kết luận.\n- **Kiểm tra:** tương quan hạng từ 0,8 trở lên và ít nhất bốn trong năm đặc trưng đầu trùng nhau.

In [ ]:
from sklearn.inspection import permutation_importance
from scipy.stats import spearmanr
s = test.sample(min(3000, len(test)), random_state=0); imps = []
for seed in range(3):                                      # the same model family with three seeds: do the explanations agree?
    m = models['LightGBM'](seed).fit(X_tr, y_tr); imps.append(permutation_importance(m, s[X_cols], s[TARGET], scoring='neg_mean_absolute_error' if IS_REG else 'roc_auc', n_repeats=3, random_state=seed).importances_mean)
imp = pd.DataFrame(np.array(imps).T, index=X_cols, columns=['seed_0', 'seed_1', 'seed_2']); imp['mean'] = imp.mean(axis=1); imp = imp.sort_values('mean', ascending=False); imp.to_csv('report/table_rankB_importance_stability.csv')
rho = float(np.mean([spearmanr(imp.seed_0, imp.seed_1)[0], spearmanr(imp.seed_0, imp.seed_2)[0], spearmanr(imp.seed_1, imp.seed_2)[0]])); top = [set(imp[c].nlargest(5).index) for c in ['seed_0', 'seed_1', 'seed_2']]
print('mean rank correlation between seeds: %.3f | features in the top five of all three seeds: %d of 5' % (rho, len(top[0] & top[1] & top[2]))); display(imp.head(8).round(4))

### R6. Bảng tổng hợp\n\n- **Làm gì:** gom năm kết quả vào một bảng để đưa vào mục Results.\n- **Kiểm tra:** ô in `advanced part: all tables written`.

In [ ]:
summary = pd.DataFrame([['best method on test', BEST, float(rk[METRIC].iloc[0])], ['runner-up', SECOND, float(rk[METRIC].iloc[1])],
                        ['gap between them has an interval that excludes zero', str(bool(bt.interval_excludes_zero.iloc[0])), float(bt.difference_a_minus_b.iloc[0])],
                        ['tuning changed LightGBM by', METRIC, float(metric(y_te, P['LightGBM tuned']) - metric(y_te, P['LightGBM']))],
                        ['explanation agreement between seeds (rank correlation)', 'LightGBM', rho]], columns=['item', 'detail', 'value'])
summary.to_csv('report/table_rankB_summary.csv', index=False); display(summary.round(4))
assert all(os.path.exists('report/table_rankB_' + n + '.csv') for n in ['tuning', 'methods', 'bootstrap', 'uncertainty', 'importance_stability', 'summary']); print('advanced part: all tables written')

## Test case và chẩn đoán

- **Làm gì:** kiểm tra tự động các sản phẩm của notebook.
- **Kiểm tra:** mọi `assert` qua. Sinh viên A chạy lại notebook này và ghi vào ô Đối chứng.

In [ ]:
assert os.path.exists('report/table_rq2.csv') and os.path.exists('report/params_best.json')
assert len(glob.glob('report/table_rq3_*.csv')) >= 2, 'RQ3 tables missing'
assert res.iloc[:, 1].notna().all()
print(sorted(os.path.basename(f) for f in glob.glob('report/*')))
print('Notebook B: all checks passed')

### Đối chứng

| Người chạy lại | Ngày | Metric chính khớp | Ghi chú |
|---|---|---|---|
| | | | |

### Chẩn đoán

Nếu mô hình tốt nhất không hơn mốc đơn giản hoặc kết quả RQ3 trái dự kiến, ghi nguyên nhân có bằng chứng (thiếu đặc trưng, dữ liệu ít, lệch giữa hai tập) vào `report/table_diagnosis.csv`.

### Báo cáo tiến độ (dán vào kênh nhóm trước buổi báo cáo)

1. Việc đã xong, kèm tên file và số dòng, bảng, hình:
2. Con số chính mới có:
3. Vướng mắc và điều đã thử:
4. Việc sẽ làm đến lần báo cáo sau, ai làm:
5. Prompt AI đáng chú ý và Human Delta: